# Transformer Summarizer Pipeline для Kaggle

Этот ноутбук содержит полный self-contained пайплайн для обучения текстового суммаризатора на базе Transformer encoder-decoder архитектуры.

Внутри ноутбука есть:

- загрузка CNN/DailyMail CSV-датасета Kaggle;
- простой tokenizer;
- Train / Validation / Test Dataset и DataLoader;
- encoder модели;
- seq2seq Transformer модель с inference;
- training loop и validation;
- графики loss/perplexity;
- тестовая генерация summary;
- сохранение checkpoint и tokenizer vocabulary в `/kaggle/working`.

Ноутбук настроен на датасет `gowrishankarp/newspaper-text-summarization-cnn-dailymail`.
Ожидаемый путь: `/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail`.
Если Kaggle смонтирует датасет иначе, ноутбук попробует найти стандартный путь автоматически.

## 1. Импорты и конфигурация

CNN/DailyMail на Kaggle обычно лежит в подпапке `cnn_dailymail` и содержит готовые split-файлы:

- `train.csv`
- `validation.csv`
- `test.csv`

Колонки датасета: `id`, `article`, `highlights`. Для обучения используем `article` как входной текст, а `highlights` как целевое summary.

In [ ]:
from __future__ import annotations

import json
import math
import random
import re
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, NamedTuple

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


@dataclass
class NotebookConfig:
    # User-provided Kaggle path. The loader also has fallbacks for Kaggle's common mount layout.
    data_dir: str = "/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail"
    dataset_subdir: str = "cnn_dailymail"
    train_file: str = "train.csv"
    validation_file: str = "validation.csv"
    test_file: str = "test.csv"
    text_column: str = "article"
    summary_column: str = "highlights"

    # Keep these limits for fast first runs. Set to None to use the full CNN/DailyMail split.
    max_train_rows: int | None = 50000
    max_validation_rows: int | None = 5000
    max_test_rows: int | None = 5000

    min_freq: int = 2
    max_vocab_size: int = 30000
    max_source_length: int = 256
    max_target_length: int = 64
    batch_size: int = 32
    gradient_accumulation_steps: int = 1
    num_workers: int = 2
    use_amp: bool = True
    use_multi_gpu: bool = True
    epochs: int = 3
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    grad_clip_norm: float = 1.0
    d_model: int = 128
    num_heads: int = 4
    num_encoder_layers: int = 2
    num_decoder_layers: int = 2
    dim_feedforward: int = 512
    dropout: float = 0.1
    output_dir: str = "/kaggle/working/summarizer"


config = NotebookConfig()
output_dir = Path(config.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"device: {device}")
print(f"cuda devices: {num_gpus}")
for gpu_idx in range(num_gpus):
    print(f"gpu {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)}")

## 2. Загрузка CNN/DailyMail датасета

Теперь используем готовые split-файлы датасета: `train.csv`, `validation.csv`, `test.csv`.
Если путь отличается от переданного `/kaggle/input/datasets/gowrishankarp/...`, функция ниже попробует стандартный Kaggle путь `/kaggle/input/newspaper-text-summarization-cnn-dailymail`.

In [ ]:
def make_demo_split() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    samples = [
        {
            "text": "Neural networks learn patterns from data and can be trained for text summarization tasks.",
            "summary": "Neural networks can summarize text.",
        },
        {
            "text": "Transformers use attention layers to model long range dependencies in sequences.",
            "summary": "Transformers model sequences with attention.",
        },
        {
            "text": "A dataloader groups tokenized examples into batches and makes training more efficient.",
            "summary": "Dataloaders batch tokenized examples.",
        },
        {
            "text": "Validation loss helps detect overfitting while the model is being trained.",
            "summary": "Validation loss tracks overfitting.",
        },
        {
            "text": "The decoder generates a summary token by token using a causal attention mask.",
            "summary": "The decoder generates summaries autoregressively.",
        },
    ]
    demo = pd.DataFrame(samples * 200)
    train_end = int(len(demo) * 0.8)
    val_end = int(len(demo) * 0.9)
    return (
        demo.iloc[:train_end].reset_index(drop=True),
        demo.iloc[train_end:val_end].reset_index(drop=True),
        demo.iloc[val_end:].reset_index(drop=True),
    )


def candidate_dataset_roots(config: NotebookConfig) -> list[Path]:
    roots = [Path(config.data_dir)]
    roots.extend(
        [
            Path("/kaggle/input/newspaper-text-summarization-cnn-dailymail"),
            Path("/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail"),
        ]
    )
    return list(dict.fromkeys(roots))


def find_split_file(config: NotebookConfig, split_file: str) -> Path | None:
    for root in candidate_dataset_roots(config):
        candidates = [
            root / config.dataset_subdir / split_file,
            root / split_file,
        ]
        for candidate in candidates:
            if candidate.exists():
                return candidate

        if root.exists():
            matches = sorted(root.rglob(split_file))
            if matches:
                return matches[0]
    return None


def normalize_split(frame: pd.DataFrame, config: NotebookConfig, limit: int | None) -> pd.DataFrame:
    required = [config.text_column, config.summary_column]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(
            f"Missing columns {missing}. Available columns: {list(frame.columns)}"
        )

    frame = frame[required].rename(
        columns={config.text_column: "text", config.summary_column: "summary"}
    )
    frame = frame.dropna().astype(str)
    frame = frame[(frame["text"].str.len() > 0) & (frame["summary"].str.len() > 0)]
    if limit is not None:
        frame = frame.head(limit)
    return frame.reset_index(drop=True)


def load_cnn_dailymail_splits(config: NotebookConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    split_specs = [
        ("train", config.train_file, config.max_train_rows),
        ("validation", config.validation_file, config.max_validation_rows),
        ("test", config.test_file, config.max_test_rows),
    ]

    loaded: dict[str, pd.DataFrame] = {}
    missing_files: list[str] = []
    for split_name, split_file, row_limit in split_specs:
        split_path = find_split_file(config, split_file)
        if split_path is None:
            missing_files.append(split_file)
            continue

        frame = pd.read_csv(split_path)
        loaded[split_name] = normalize_split(frame, config, row_limit)
        print(f"{split_name}: {split_path} -> {len(loaded[split_name])} rows")

    if missing_files:
        print(f"Could not find split files: {missing_files}. Falling back to demo dataset.")
        return make_demo_split()

    return loaded["train"], loaded["validation"], loaded["test"]


train_df, val_df, test_df = load_cnn_dailymail_splits(config)
print({"train": len(train_df), "validation": len(val_df), "test": len(test_df)})
train_df.head()

## 3. Tokenizer

Для учебного проекта используется простой word-level tokenizer. Для более сильного качества позже можно заменить его на BPE/SentencePiece tokenizer, не меняя общую идею Dataset/DataLoader и модели.

In [ ]:
class SimpleTokenizer:
    pad_token = "<pad>"
    bos_token = "<bos>"
    eos_token = "<eos>"
    unk_token = "<unk>"

    def __init__(self, min_freq: int = 2, max_vocab_size: int = 30000) -> None:
        self.min_freq = min_freq
        self.max_vocab_size = max_vocab_size
        self.token_to_id = {
            self.pad_token: 0,
            self.bos_token: 1,
            self.eos_token: 2,
            self.unk_token: 3,
        }
        self.id_to_token = {idx: token for token, idx in self.token_to_id.items()}

    @property
    def pad_token_id(self) -> int:
        return self.token_to_id[self.pad_token]

    @property
    def bos_token_id(self) -> int:
        return self.token_to_id[self.bos_token]

    @property
    def eos_token_id(self) -> int:
        return self.token_to_id[self.eos_token]

    @property
    def unk_token_id(self) -> int:
        return self.token_to_id[self.unk_token]

    @property
    def vocab_size(self) -> int:
        return len(self.token_to_id)

    @staticmethod
    def tokenize(text: str) -> list[str]:
        text = text.lower().strip()
        return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

    def fit(self, texts: list[str]) -> None:
        counts: Counter[str] = Counter()
        for text in texts:
            counts.update(self.tokenize(text))

        max_new_tokens = self.max_vocab_size - len(self.token_to_id)
        for token, count in counts.most_common(max_new_tokens):
            if count < self.min_freq:
                continue
            if token not in self.token_to_id:
                idx = len(self.token_to_id)
                self.token_to_id[token] = idx
                self.id_to_token[idx] = token

    def ids(self, text: str) -> list[int]:
        return [self.token_to_id.get(token, self.unk_token_id) for token in self.tokenize(text)]

    def pad(self, token_ids: list[int], max_length: int) -> list[int]:
        token_ids = token_ids[:max_length]
        return token_ids + [self.pad_token_id] * (max_length - len(token_ids))

    def encode_source(self, text: str, max_length: int) -> list[int]:
        token_ids = self.ids(text)[: max_length - 1] + [self.eos_token_id]
        return self.pad(token_ids, max_length)

    def encode_target_pair(self, text: str, max_length: int) -> tuple[list[int], list[int]]:
        token_ids = self.ids(text)[: max_length - 1]
        decoder_input = [self.bos_token_id] + token_ids
        labels = token_ids + [self.eos_token_id]
        return self.pad(decoder_input, max_length), self.pad(labels, max_length)

    def decode(self, token_ids: list[int]) -> str:
        skip = {self.pad_token_id, self.bos_token_id, self.eos_token_id}
        tokens = [self.id_to_token.get(int(idx), self.unk_token) for idx in token_ids if int(idx) not in skip]
        text = " ".join(tokens)
        text = re.sub(r"\s+([.,!?;:])", r"", text)
        return text.strip()

    def to_dict(self) -> dict[str, Any]:
        return {
            "min_freq": self.min_freq,
            "max_vocab_size": self.max_vocab_size,
            "token_to_id": self.token_to_id,
        }


tokenizer = SimpleTokenizer(min_freq=config.min_freq, max_vocab_size=config.max_vocab_size)
tokenizer.fit((train_df["text"].tolist() + train_df["summary"].tolist()))
print(f"vocab_size: {tokenizer.vocab_size}")

## 3.1. Анализ длины текстов в токенах

Максимальная длина зависит от tokenizer-а. Эта ячейка считает длины `article` и `highlights` уже нашим `SimpleTokenizer`, чтобы выбрать разумные `max_source_length` и `max_target_length`.

In [ ]:
def token_length_stats(frame: pd.DataFrame, column: str, sample_name: str) -> dict[str, float]:
    lengths = frame[column].map(lambda text: len(tokenizer.tokenize(text)))
    stats = {
        "split": sample_name,
        "column": column,
        "rows": int(len(lengths)),
        "mean": float(lengths.mean()),
        "p50": float(lengths.quantile(0.50)),
        "p90": float(lengths.quantile(0.90)),
        "p95": float(lengths.quantile(0.95)),
        "p99": float(lengths.quantile(0.99)),
        "max": int(lengths.max()),
    }
    return stats


length_stats = []
for split_name, frame in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    length_stats.append(token_length_stats(frame, "text", split_name))
    length_stats.append(token_length_stats(frame, "summary", split_name))

length_stats_df = pd.DataFrame(length_stats)
display(length_stats_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_df["text"].map(lambda text: len(tokenizer.tokenize(text))).hist(bins=60, ax=axes[0])
axes[0].axvline(config.max_source_length, color="red", linestyle="--", label=f"max_source_length={config.max_source_length}")
axes[0].set_title("Train article token lengths")
axes[0].set_xlabel("tokens")
axes[0].legend()

train_df["summary"].map(lambda text: len(tokenizer.tokenize(text))).hist(bins=60, ax=axes[1])
axes[1].axvline(config.max_target_length, color="red", linestyle="--", label=f"max_target_length={config.max_target_length}")
axes[1].set_title("Train summary token lengths")
axes[1].set_xlabel("tokens")
axes[1].legend()
plt.show()

## 4. Dataset и DataLoader

Каждый batch возвращает три тензора:

- `src_tokens`: токены исходного текста;
- `tgt_tokens`: decoder input, начинается с `<bos>`;
- `labels`: target summary, сдвинутый на один токен и заканчивающийся `<eos>`.

In [ ]:
class SummarizationDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer: SimpleTokenizer,
        max_source_length: int,
        max_target_length: int,
    ) -> None:
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, Tensor]:
        row = self.frame.iloc[index]
        src_tokens = self.tokenizer.encode_source(row["text"], self.max_source_length)
        tgt_tokens, labels = self.tokenizer.encode_target_pair(row["summary"], self.max_target_length)
        return {
            "src_tokens": torch.tensor(src_tokens, dtype=torch.long),
            "tgt_tokens": torch.tensor(tgt_tokens, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def make_dataloader(frame: pd.DataFrame, shuffle: bool) -> DataLoader:
    dataset = SummarizationDataset(
        frame=frame,
        tokenizer=tokenizer,
        max_source_length=config.max_source_length,
        max_target_length=config.max_target_length,
    )
    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=shuffle,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available(),
    )


train_loader = make_dataloader(train_df, shuffle=True)
val_loader = make_dataloader(val_df, shuffle=False)
test_loader = make_dataloader(test_df, shuffle=False)

batch = next(iter(train_loader))
print({key: tuple(value.shape) for key, value in batch.items()})

## 5. Encoder и Transformer Summarizer

Ниже находится архитектура нейросети: отдельный encoder, decoder, causal mask и inference через greedy decoding.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_length: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        positions = torch.arange(max_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))
        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = torch.sin(positions * div_term)
        encoding[:, 1::2] = torch.cos(positions * div_term)
        self.register_buffer("encoding", encoding.unsqueeze(0), persistent=False)

    def forward(self, token_embeddings: Tensor) -> Tensor:
        sequence_length = token_embeddings.size(1)
        position_encoding = self.encoding[:, :sequence_length].to(dtype=token_embeddings.dtype)
        return self.dropout(token_embeddings + position_encoding)


class TransformerTextEncoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        pad_token_id: int,
        max_length: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.pad_token_id = pad_token_id
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        self.positions = SinusoidalPositionalEncoding(d_model, max_length, dropout)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers, norm=nn.LayerNorm(d_model))

    def make_padding_mask(self, tokens: Tensor) -> Tensor:
        return tokens.eq(self.pad_token_id)

    def forward(self, input_ids: Tensor, key_padding_mask: Tensor | None = None) -> Tensor:
        if key_padding_mask is None:
            key_padding_mask = self.make_padding_mask(input_ids)
        embeddings = self.token_embedding(input_ids) * math.sqrt(self.d_model)
        embeddings = self.positions(embeddings)
        return self.encoder(embeddings, src_key_padding_mask=key_padding_mask)


@dataclass
class TransformerSummarizerConfig:
    vocab_size: int
    pad_token_id: int
    max_source_length: int
    max_target_length: int
    d_model: int
    num_heads: int
    num_encoder_layers: int
    num_decoder_layers: int
    dim_feedforward: int
    dropout: float = 0.1


class SummarizerOutput(NamedTuple):
    logits: Tensor
    encoder_last_hidden_state: Tensor
    decoder_last_hidden_state: Tensor
    loss: Tensor | None = None


class TransformerTextSummarizer(nn.Module):
    def __init__(self, model_config: TransformerSummarizerConfig) -> None:
        super().__init__()
        self.config = model_config
        self.encoder = TransformerTextEncoder(
            vocab_size=model_config.vocab_size,
            pad_token_id=model_config.pad_token_id,
            max_length=model_config.max_source_length,
            d_model=model_config.d_model,
            num_heads=model_config.num_heads,
            num_layers=model_config.num_encoder_layers,
            dim_feedforward=model_config.dim_feedforward,
            dropout=model_config.dropout,
        )
        self.tgt_embedding = nn.Embedding(model_config.vocab_size, model_config.d_model, padding_idx=model_config.pad_token_id)
        self.target_positions = SinusoidalPositionalEncoding(model_config.d_model, model_config.max_target_length, model_config.dropout)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=model_config.d_model,
            nhead=model_config.num_heads,
            dim_feedforward=model_config.dim_feedforward,
            dropout=model_config.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=model_config.num_decoder_layers, norm=nn.LayerNorm(model_config.d_model))
        self.lm_head = nn.Linear(model_config.d_model, model_config.vocab_size, bias=False)
        self.tgt_embedding.weight = self.encoder.token_embedding.weight
        self.lm_head.weight = self.tgt_embedding.weight

    def make_padding_mask(self, tokens: Tensor) -> Tensor:
        return tokens.eq(self.config.pad_token_id)

    @staticmethod
    def causal_mask(size: int, device: torch.device) -> Tensor:
        return torch.triu(torch.ones(size, size, dtype=torch.bool, device=device), diagonal=1)

    def encode(self, src_tokens: Tensor, src_key_padding_mask: Tensor | None = None) -> Tensor:
        return self.encoder(src_tokens, key_padding_mask=src_key_padding_mask)

    def decode(
        self,
        tgt_tokens: Tensor,
        memory: Tensor,
        memory_key_padding_mask: Tensor | None = None,
        tgt_key_padding_mask: Tensor | None = None,
    ) -> Tensor:
        if tgt_key_padding_mask is None:
            tgt_key_padding_mask = self.make_padding_mask(tgt_tokens)
        embeddings = self.tgt_embedding(tgt_tokens) * math.sqrt(self.config.d_model)
        embeddings = self.target_positions(embeddings)
        return self.decoder(
            tgt=embeddings,
            memory=memory,
            tgt_mask=self.causal_mask(tgt_tokens.size(1), tgt_tokens.device),
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )

    def forward(self, src_tokens: Tensor, tgt_tokens: Tensor, labels: Tensor | None = None) -> SummarizerOutput:
        src_key_padding_mask = self.make_padding_mask(src_tokens)
        memory = self.encode(src_tokens, src_key_padding_mask=src_key_padding_mask)
        decoder_hidden = self.decode(tgt_tokens, memory, memory_key_padding_mask=src_key_padding_mask)
        logits = self.lm_head(decoder_hidden)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=self.config.pad_token_id)
        return SummarizerOutput(logits=logits, encoder_last_hidden_state=memory, decoder_last_hidden_state=decoder_hidden, loss=loss)

    @torch.no_grad()
    def generate(self, src_tokens: Tensor, bos_token_id: int, eos_token_id: int, max_new_tokens: int) -> Tensor:
        self.eval()
        src_key_padding_mask = self.make_padding_mask(src_tokens)
        memory = self.encode(src_tokens, src_key_padding_mask=src_key_padding_mask)
        generated = torch.full((src_tokens.size(0), 1), bos_token_id, dtype=torch.long, device=src_tokens.device)
        finished = torch.zeros(src_tokens.size(0), dtype=torch.bool, device=src_tokens.device)
        for _ in range(max_new_tokens):
            decoder_hidden = self.decode(generated, memory, memory_key_padding_mask=src_key_padding_mask)
            next_token = self.lm_head(decoder_hidden[:, -1]).argmax(dim=-1)
            next_token = torch.where(finished, torch.full_like(next_token, self.config.pad_token_id), next_token)
            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            finished |= next_token.eq(eos_token_id)
            if finished.all():
                break
        return generated

## 6. Training / Evaluation / Графики

## 6.1. Multi-GPU и экономия памяти

Для Kaggle с двумя Tesla T4 16GB используется `nn.DataParallel`: модель копируется на обе GPU, а batch делится между ними. Это увеличивает эффективный batch size, но не увеличивает максимальную длину одной последовательности. Если будет CUDA OOM, сначала уменьшайте `batch_size`, `max_source_length`, `max_target_length`, `d_model` или включайте `gradient_accumulation_steps > 1`.

In [ ]:
def move_batch_to_device(batch: dict[str, Tensor], device: torch.device) -> dict[str, Tensor]:
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


amp_enabled = config.use_amp and device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
print(f"mixed precision AMP: {amp_enabled}")
print(f"gradient_accumulation_steps: {config.gradient_accumulation_steps}")


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
) -> float:
    model.train()
    total_loss = 0.0
    total_batches = 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, start=1):
        batch = move_batch_to_device(batch, device)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            output = model(batch["src_tokens"], batch["tgt_tokens"], labels=batch["labels"])
            loss = output.loss / config.gradient_accumulation_steps

        scaler.scale(loss).backward()

        should_step = step % config.gradient_accumulation_steps == 0 or step == len(loader)
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += float(output.loss.detach().cpu())
        total_batches += 1

    return total_loss / max(total_batches, 1)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0
    total_batches = 0
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            output = model(batch["src_tokens"], batch["tgt_tokens"], labels=batch["labels"])
        total_loss += float(output.loss.detach().cpu())
        total_batches += 1
    return total_loss / max(total_batches, 1)


def perplexity(loss: float) -> float:
    return math.exp(min(loss, 20.0))


def plot_history(history: dict[str, list[float]]) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
    axes[0].plot(epochs, history["val_loss"], marker="o", label="validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, [perplexity(x) for x in history["train_loss"]], marker="o", label="train")
    axes[1].plot(epochs, [perplexity(x) for x in history["val_loss"]], marker="o", label="validation")
    axes[1].set_title("Perplexity")
    axes[1].set_xlabel("epoch")
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    plt.show()

## 7. Создание и тренировка модели

In [ ]:
model_config = TransformerSummarizerConfig(
    vocab_size=tokenizer.vocab_size,
    pad_token_id=tokenizer.pad_token_id,
    max_source_length=config.max_source_length,
    max_target_length=config.max_target_length,
    d_model=config.d_model,
    num_heads=config.num_heads,
    num_encoder_layers=config.num_encoder_layers,
    num_decoder_layers=config.num_decoder_layers,
    dim_feedforward=config.dim_feedforward,
    dropout=config.dropout,
)

model = TransformerTextSummarizer(model_config).to(device)
if config.use_multi_gpu and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"using DataParallel on {torch.cuda.device_count()} GPUs")
else:
    print("using single device training")

optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, config.epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
    val_loss = evaluate(model, val_loader)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_loss:.4f} train_ppl={perplexity(train_loss):.2f} "
        f"val_loss={val_loss:.4f} val_ppl={perplexity(val_loss):.2f}"
    )

plot_history(history)

## 8. Тестирование модели и inference

После обучения проверяем test loss и генерируем несколько summary. На demo dataset качество будет игрушечным; на реальном датасете оно зависит от размера данных, tokenizer и числа эпох.

In [ ]:
@torch.no_grad()
def summarize(text: str, max_new_tokens: int | None = None) -> str:
    inference_model = unwrap_model(model)
    inference_model.eval()
    max_new_tokens = max_new_tokens or config.max_target_length - 1
    src_tokens = tokenizer.encode_source(text, config.max_source_length)
    src_tensor = torch.tensor([src_tokens], dtype=torch.long, device=device)
    generated = inference_model.generate(
        src_tensor,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=max_new_tokens,
    )
    return tokenizer.decode(generated[0].detach().cpu().tolist())


test_loss = evaluate(model, test_loader)
print(f"test_loss={test_loss:.4f}, test_ppl={perplexity(test_loss):.2f}")

examples = test_df.head(5).copy()
examples["generated_summary"] = examples["text"].apply(summarize)
examples[["text", "summary", "generated_summary"]]

## 9. Сохранение артефактов

Kaggle сохраняет файлы из `/kaggle/working`. После выполнения ноутбука можно скачать checkpoint и vocabulary.

In [ ]:
checkpoint_path = output_dir / "summarizer_checkpoint.pt"
vocab_path = output_dir / "tokenizer_vocab.json"
history_path = output_dir / "history.json"

torch.save(
    {
        "model_config": asdict(model_config),
        "model_state_dict": unwrap_model(model).state_dict(),
        "tokenizer": tokenizer.to_dict(),
        "history": history,
    },
    checkpoint_path,
)

with vocab_path.open("w", encoding="utf-8") as file:
    json.dump(tokenizer.to_dict(), file, ensure_ascii=False, indent=2)

with history_path.open("w", encoding="utf-8") as file:
    json.dump(history, file, ensure_ascii=False, indent=2)

print(f"saved checkpoint: {checkpoint_path}")
print(f"saved tokenizer: {vocab_path}")
print(f"saved history: {history_path}")